# Retrieval-Augmented Generation (RAG) System
### Week Assignment — Custom Document Question Answering

This notebook builds a complete **RAG pipeline** that answers questions grounded in a
custom set of documents (PDF/TXT), instead of relying only on the language model's
internal knowledge.

**Pipeline stages covered:**
1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database (FAISS)
5. Query Processing
6. Context Retrieval
7. Answer Generation


## 1. Install Dependencies

In [ ]:
# Run this once. Restart the kernel if prompted after install.
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch


## 2. Imports

In [ ]:
import os
import glob
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("All libraries imported successfully.")


## 3. Configuration

- `DATA_DIR` — folder containing your `.pdf` / `.txt` documents.
- `EMBED_MODEL_NAME` — a small, fast, free embedding model.
- `GEN_MODEL_NAME` — a small, free, local text-generation model (no API key needed).
- `CHUNK_SIZE` / `CHUNK_OVERLAP` — controls how documents are split.
- `TOP_K` — number of chunks retrieved per query.

> If you have your own dataset, just drop `.pdf` or `.txt` files into `DATA_DIR` and
> re-run the notebook from Section 4 onward. See the end of this notebook for
> **where to get a dataset**.


In [ ]:
DATA_DIR = "data"                          # folder with your documents
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"      # sentence-transformers embedding model
GEN_MODEL_NAME = "google/flan-t5-base"     # free local generation model
CHUNK_SIZE = 500                            # characters per chunk
CHUNK_OVERLAP = 50                          # overlap between chunks
TOP_K = 3                                   # number of chunks to retrieve

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Using data directory: {os.path.abspath(DATA_DIR)}")


## 4. Stage 1 — Document Ingestion

Loads every `.pdf` and `.txt` file inside `DATA_DIR` and converts it into raw text.


In [ ]:
def load_pdf(path):
    text = ""
    reader = PdfReader(path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def load_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def load_documents(data_dir):
    """Loads all .pdf and .txt files from data_dir into {filename: text} dict."""
    documents = {}
    filepaths = glob.glob(os.path.join(data_dir, "*.pdf")) + \
                glob.glob(os.path.join(data_dir, "*.txt"))

    if not filepaths:
        raise FileNotFoundError(
            f"No .pdf or .txt files found in '{data_dir}'. Add documents and re-run."
        )

    for path in filepaths:
        filename = os.path.basename(path)
        if path.lower().endswith(".pdf"):
            documents[filename] = load_pdf(path)
        else:
            documents[filename] = load_txt(path)
        print(f"Loaded: {filename}  ({len(documents[filename])} characters)")

    return documents


raw_documents = load_documents(DATA_DIR)
print(f"\nTotal documents loaded: {len(raw_documents)}")


## 5. Stage 2 — Text Chunking

Splits each document into overlapping fixed-size chunks so retrieval can pinpoint
the most relevant piece of text rather than an entire document.


In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Splits text into overlapping chunks of `chunk_size` characters."""
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


all_chunks = []       # list of chunk strings
chunk_sources = []    # parallel list: which file each chunk came from

for filename, text in raw_documents.items():
    doc_chunks = chunk_text(text)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([filename] * len(doc_chunks))

print(f"Total chunks created: {len(all_chunks)}")
print(f"\nExample chunk:\n{all_chunks[0][:300]}...")


## 6. Stage 3 — Embedding Creation

Converts every chunk into a dense vector using a Sentence-Transformers model. The
first run downloads the model (~90MB), so an internet connection is required once.


In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

print(f"Embedding matrix shape: {chunk_embeddings.shape}")


## 7. Stage 4 — Vector Database (FAISS)

Stores the chunk embeddings in a FAISS index for fast similarity search.


In [ ]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)   # simple, exact L2 similarity search
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")


## 8. Stage 5 & 6 — Query Processing and Context Retrieval

Embeds the user's question and retrieves the `TOP_K` most similar chunks from the
FAISS index.


In [ ]:
def retrieve_context(query, top_k=TOP_K):
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk": all_chunks[idx],
            "source": chunk_sources[idx],
            "distance": float(distances[0][rank])
        })
    return results


# Quick test
test_results = retrieve_context("What is RAG used for?")
for r in test_results:
    print(f"[{r['source']} | distance={r['distance']:.4f}]")
    print(r["chunk"][:200], "...\n")


## 9. Stage 7 — Answer Generation

Loads a small, free, local generation model (`google/flan-t5-base`) and uses the
retrieved chunks as grounding context to answer the question. No API key required.


In [ ]:
# NOTE: We load the tokenizer/model directly instead of using transformers.pipeline().
# Newer versions of `transformers` (v5+) removed the "text2text-generation" pipeline
# task, and the generic "text-generation" pipeline only supports causal (decoder-only)
# models -- not encoder-decoder models like flan-t5. Loading the model directly with
# AutoModelForSeq2SeqLM works reliably across all transformers versions.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([c["chunk"] for c in retrieved_chunks])
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


def generate_answer(query, top_k=TOP_K, max_new_tokens=200):
    retrieved = retrieve_context(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)

    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return answer, retrieved


## 10. Full RAG Pipeline — Demo

In [ ]:
def rag_answer(query):
    answer, sources = generate_answer(query)
    print(f"Q: {query}\n")
    print(f"A: {answer}\n")
    print("Sources used:")
    for s in sources:
        print(f"  - {s['source']} (distance={s['distance']:.4f})")
    print("-" * 60)
    return answer


# Try a few sample questions
rag_answer("What is Retrieval-Augmented Generation?")
rag_answer("What is AI?")
rag_answer("Why is chunk overlap important?")


## 11. Try Your Own Question

Change the `my_question` variable below and re-run the cell.


In [ ]:
my_question = "What embedding model is commonly used for RAG?"
rag_answer(my_question)


## 12. Summary

This notebook implemented a full RAG pipeline:

| Stage | Tool Used |
|---|---|
| Document Ingestion | `pypdf`, plain file I/O |
| Text Chunking | Custom fixed-size overlapping chunker |
| Embedding Creation | `sentence-transformers` (`all-MiniLM-L6-v2`) |
| Vector Database | `faiss` (`IndexFlatL2`) |
| Query Processing | Same embedding model, encoded at query time |
| Context Retrieval | FAISS similarity search (top-k) |
| Answer Generation | `transformers` pipeline (`google/flan-t5-base`) |

**Possible improvements / extensions to mention in your report:**
- Swap `flan-t5-base` for a larger model (e.g. via an API) for higher-quality answers.
- Use `IndexHNSWFlat` or `IndexIVFFlat` in FAISS for faster search on large corpora.
- Add re-ranking of retrieved chunks with a cross-encoder for better precision.
- Track answer relevance with a small eval set (question + expected answer pairs).
